## Import

In [244]:
import io
import numpy as np
import pandas as pd
from pandas.tseries.offsets import Week, Day
from dateutil.easter import easter

## Metadata

In [245]:
# Read the file line by line
file_path = 'data/All_Residential.txt'

with open(file_path, 'r', encoding="utf-8") as f:
    lines = f.read().splitlines() # read line by line

In [246]:
# Keep only the table (everything before the first "***")
end = next((i for i, line in enumerate(lines) if line.strip() == "***"), None) # find the index of element that start with ***
table_lines = [line for line in lines[:end] if line.strip()] # remove the empty space line and the line with
table_str = '\n'.join(table_lines) #convert list to string
table_io = io.StringIO(table_str) # convert to string
df = pd.read_fwf(table_io)
metadata = df.drop([0]) # drop the line below header

In [247]:
# clean up metadata
metadata['HouseType_missing'] = metadata['HouseType'].isna().astype(int) # check for null
metadata['HouseType'] = metadata['HouseType'].fillna(metadata['HouseType'].mode()[0]) # fill in nans with most common house  

metadata['Facing_missing'] = metadata['Facing'].isna().astype(int) # check for null
metadata['Facing'] = metadata['Facing'].fillna(metadata['Facing'].mode()[0]) # fill in nans with most common facing direction

metadata['Region_missing'] = metadata['Region'].isna().astype(int) # check for null
metadata['Region'] = metadata['Region'].fillna(metadata['Region'].mode()[0]) # fill in nans with most common region

metadata['HVAC_missing'] = metadata['HVAC'].isna().astype(int) # check for null
metadata['HVAC'] = metadata['HVAC'].fillna(metadata['HVAC'].mode()[0]) # fill in nans with most common HVAC

In [248]:
metadata['House'] = metadata['House'].astype(int) # for merging with all residential data later
display(metadata)

,House,FirstReading,LastReading,Cover,HouseType,Facing,Region,RUs,EVs,SN,HVAC,HouseType_missing,Facing_missing,Region_missing,HVAC_missing
1,1,2012-06-01,2015-10-03,1.000,bungalow,South,YVR,1,NaN,1,FAGF + FPG + HP,0,0,0,0
2,2,2016-06-09,2019-11-20,0.994,duplex,North,YVR,0,NaN,2,"IFRHG + FPG, NAC",0,0,0,0
3,3,2015-01-27,2018-01-29,0.987,modern,South,YVR,2,NaN,NaN,"IFRHG + 1 BHE, NAC",0,0,0,0
4,4,2015-01-30,2018-01-29,0.995,character,West,YVR,1,NaN,NaN,"FAGF + IFRHG, NAC",0,0,0,0
5,5,2015-01-30,2018-01-29,0.995,modern,South,YVR,1,NaN,NaN,"IFRHG, NAC",0,0,0,0
6,6,2015-01-30,2018-01-29,0.997,apartment,SW,YVR,0,NaN,NaN,"BHE, NAC",0,0,0,0
7,7,2015-01-30,2018-01-29,0.997,apartment,South,YVR,NaN,NaN,NaN,"BHE, NAC",1,1,1,1
8,8,2015-02-21,2018-02-20,0.987,character,South,YVR,0,NaN,NaN,"FAGF, PAC",0,0,0,0
9,9,2015-05-01,2018-02-21,0.996,special,South,YVR,0,NaN,NaN,"IFRHG + FPG, NAC",0,0,0,0
10,10,2015-02-21,2018-02-20,0.995,special,South,YVR,0,NaN,NaN,"FAGF, NAC",0,0,0,0


## Holiday Data

In [249]:
# read file
holidays = pd.read_csv('data/Holidays.csv')

In [250]:
holidays['date'] = pd.to_datetime(holidays['date']) # convert string to datetime data type

In [251]:
def first_weekday(year, month, weekday): # find that specific first day of the month based on the weekday index below
    """
    weekday:
        Monday=0
        Tuesday=1
        ...
        Sunday=6
    """
    d = pd.Timestamp(year, month, 1)

    while d.weekday() != weekday:
        d += Day(1)

    return d

#n, says which occurrence of that day you want: 1 for the first, 2 for the second, and so on.
def nth_weekday(year, month, weekday, n): # find that specific n-th day of the month based on the weekday index
    return first_weekday(year, month, weekday) + Week(n-1)

def last_weekday(year, month, weekday): # find that specific last day of the month based on the weekday index

    d = pd.Timestamp(year, month + 1, 1) - Day(1)

    while d.weekday() != weekday:
        d -= Day(1)

    return d

In [252]:
def canadian_holidays(year):

    easter_sunday = pd.Timestamp(easter(year))

    dict_holidays = {
        "New Years":
            pd.Timestamp(year,1,1),
        "Good Friday":
            easter_sunday - Day(2),
        "Easter Monday":
            easter_sunday + Day(1),
        "Victoria Day":
            last_weekday(year,5,0) - Week(1),
        "Canada Day":
            pd.Timestamp(year,7,1),
        "Civic Day":
            first_weekday(year,8,0),
        "Labour Day":
            first_weekday(year,9,0),
        "Thanksgiving":
            nth_weekday(year,10,0,2),
        "Remembrance Day":
            pd.Timestamp(year,11,11),
        "Christmas Day":
            pd.Timestamp(year,12,25),
        "Boxing Day":
            pd.Timestamp(year,12,26),
        "Family Day":
            nth_weekday(year,2,0,3)
    }
    return dict_holidays

In [253]:
def merge_holidays(df_in):
    
    years = range(
        df_in['date'].dt.year.min(),
        df_in['date'].dt.year.max()+1
    )

    holiday_rows = []

    for year in years:
        dict_holidays = canadian_holidays(year)
        for name, date in dict_holidays.items():
            holiday_rows.append({"date": date,"holiday": name})
            
    df_holidays = pd.DataFrame(holiday_rows)

    df_out = pd.merge(df_in, df_holidays, on='date', how='left')
    df_out['holiday'] = df_out['holiday'].fillna('None')
    
    return df_out

## Energy Household Data

In [254]:
# Load energy consumption data for each house
energy_data = []
for i in range(1, 29):
    file_path = f'data/Residential_{i}.csv'
    df = pd.read_csv(file_path)
    df['date'] = pd.to_datetime(df['date'])
    
    # change hour to be 1 indexed instead of 0 indexed
    df['hour'] += 1

    # fix daylight savings shifts: Daylight saving starts in Mar and ends in Nov
    df['dst'] = 0
    # the start of daylight saving on the 2nd sunday of Mar => missing hour
    gap_mask = df['hour'].diff() == 2
    gap_mask &= df['date'].dt.month == 3
    gap_indices = list(np.where(gap_mask==True)[0])  # get indexes of those missings

    # the end of daylight saving on the 1st Sunday of Nov => repeated hour
    repeat_mask = df['hour'].diff() == 0
    repeat_mask &= df['date'].dt.month == 11
    repeat_indices = list(np.where(repeat_mask==True)[0]) # get indexes of those repeated hours

    # if that Residential data has only the end of daylight saving but not the start of daylight saving
    if (len(repeat_indices) == 1) and (len(gap_indices) == 0):
        gap_indices.insert(0,0) # in the current dataset, although there is no start of daylight saving in Mar, the current start date and time is already in affective of Mar logics

    # if the dataset has Mar start of daylight saving but not the end of it
    if (len(gap_indices) == 1) and (len(repeat_indices) == 0):
        repeat_indices.append(len(df)) # the November transition is missing, perhaps because the dataset ends before November so from the index of the start of Daylight saving until the last index in that data is still in Mar logics. Anything outside that is Nov logics

    # speaking of the index order, if the first Nov index is lesser than the first Mar index, that means the data start with Nov first
    if repeat_indices[0] < gap_indices[0]:
        gap_indices.insert(0,0) # that means before reaching the first Nov, it has already been following the Mar logic since the first index

    # speaking of the index order, if the last Nov index is lesser than the last Mar index, that means the data end with Mar 
    if repeat_indices[-1] < gap_indices[-1]:
        repeat_indices.append(len(df)) # that means from the start of the last Nov until the last indexes, it still follows Nov logics

    # for each DST period (start -> stop): shift hours back to standard time and flag dst = 1
    for start, stop in zip(gap_indices, repeat_indices): # pair start and last index with the if conditions above
        df.loc[start:stop-1, 'hour'] -= 1 # that missing hour is handled by substracting everything by 1 and then everything is converted to the standard time so no hour is using Daylight saving standard
        df.loc[start:stop-1, 'dst'] = 1
        
    # fix hours that are 0
    zero_mask = df['hour'] == 0
    df.loc[zero_mask, 'hour'] = 24 # find hours that are 0 after the subtraction above and assign it to 24h
    df.loc[zero_mask, 'date'] -= pd.Timedelta(days=1) # need to adjust the date for those hours
    
    # get rid of duplicate rows
    df = df.drop_duplicates()
    
    # clean up data
    df['datetime'] = pd.to_datetime(df['date'])+pd.to_timedelta(df['hour']-1, unit='h') # get the date time from data
    df = df.set_index('datetime') # make datetime as the index of the dataframe
    df = df.reindex(pd.date_range(df.index.min(), df.index.max(), freq='h')) # build a complete hour between the start date and end date. If there is any missing hour or day, it will assign null to the daate and hour fields
    df['date'] = df.index.strftime("%Y-%m-%d") # reassign the date in case there is any missing date
    df['hour'] = df.index.hour+1 # reassign the hour in case there is any missing hour and +1 is added so that the hour start with 1 rather than 0
    df = df.reset_index(drop=True)
    
    # handle null value for energy_kWh field
    df['kWh_missing'] = df['energy_kWh'].isna().astype(int)
    column_list = ['energy_kWh']
    
    for column in column_list:
        selected_energy_feature = df[column]

        # average of the hour before and after
        energy_feature = (selected_energy_feature.shift(1) + selected_energy_feature.shift(-1)) / 2

        # if energy_feature is still null, average of the same hour one day before and after
        energy_feature = energy_feature.fillna((selected_energy_feature.shift(24) + selected_energy_feature.shift(-24)) / 2)

        # if energy_feature is still null, copy the same hour from the day before
        energy_feature = energy_feature.fillna(selected_energy_feature.shift(24))

        # if energy_feature is still null, copy the same hour from the day after
        energy_feature = energy_feature.fillna(selected_energy_feature.shift(-24))

        # write back: only fill the original NaN rows
        df[column] = selected_energy_feature.fillna(energy_feature.round(3))
            
    df['House'] = i
    df['date'] = pd.to_datetime(df['date'])    
    
    energy_data.append(df)

energy_data = pd.concat(energy_data, ignore_index=True)

# Convert House column to int64
energy_data['House'] = energy_data['House'].astype(int)

# set a minimum baseline energy usage where there are hours with 0
energy_data.loc[energy_data['energy_kWh']==0,'energy_kWh'] = energy_data[energy_data['energy_kWh'] > 0]['energy_kWh'].min()

In [255]:
# Merge energy data with metadata
merged_residential_data = pd.merge(energy_data, metadata, on='House', how='left')

In [256]:
display(merged_residential_data)

,date,hour,energy_kWh,dst,kWh_missing,House,FirstReading,LastReading,Cover,HouseType,Facing,Region,RUs,EVs,SN,HVAC,HouseType_missing,Facing_missing,Region_missing,HVAC_missing
0,2012-06-01,1,1.011,1.0,0,1,2012-06-01,2015-10-03,1.000,bungalow,South,YVR,1,NaN,1,FAGF + FPG + HP,0,0,0,0
1,2012-06-01,2,0.451,1.0,0,1,2012-06-01,2015-10-03,1.000,bungalow,South,YVR,1,NaN,1,FAGF + FPG + HP,0,0,0,0
2,2012-06-01,3,0.505,1.0,0,1,2012-06-01,2015-10-03,1.000,bungalow,South,YVR,1,NaN,1,FAGF + FPG + HP,0,0,0,0
3,2012-06-01,4,0.441,1.0,0,1,2012-06-01,2015-10-03,1.000,bungalow,South,YVR,1,NaN,1,FAGF + FPG + HP,0,0,0,0
4,2012-06-01,5,0.468,1.0,0,1,2012-06-01,2015-10-03,1.000,bungalow,South,YVR,1,NaN,1,FAGF + FPG + HP,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
616075,2020-05-19,19,3.060,1.0,0,28,2018-08-01,2020-05-19,0.998,special,North,YVR,0,NaN,NaN,"FAGF + 1 FPE + 2 FPG, NAC",0,0,0,0
616076,2020-05-19,20,2.940,1.0,0,28,2018-08-01,2020-05-19,0.998,special,North,YVR,0,NaN,NaN,"FAGF + 1 FPE + 2 FPG, NAC",0,0,0,0
616077,2020-05-19,21,1.970,1.0,0,28,2018-08-01,2020-05-19,0.998,special,North,YVR,0,NaN,NaN,"FAGF + 1 FPE + 2 FPG, NAC",0,0,0,0
616078,2020-05-19,22,2.130,1.0,0,28,2018-08-01,2020-05-19,0.998,special,North,YVR,0,NaN,NaN,"FAGF + 1 FPE + 2 FPG, NAC",0,0,0,0


## Weather Data

In [258]:
# Load YVR weather data
weather_yvr = pd.read_csv('data/Weather_YVR.csv')

In [259]:
# clean up YVR data
weather_yvr['datetime'] = pd.to_datetime(weather_yvr['date'])+pd.to_timedelta(weather_yvr['hour']-1, unit='h') # get the date time from data
weather_yvr = weather_yvr.set_index('datetime') # make datetime as the index of the dataframe
weather_yvr = weather_yvr.reindex(pd.date_range(weather_yvr.index.min(), weather_yvr.index.max(), freq='h')) # build a complete hour between the start date and end date. If there is any missing hour or day, it will assign null to the daate and hour fields
weather_yvr['date'] = weather_yvr.index.strftime("%Y-%m-%d") # reassign the date in case there is any missing date
weather_yvr['hour'] = weather_yvr.index.hour+1 # reassign the hour in case there is any missing hour and +1 is added so that the hour start with 1 rather than 0
weather_yvr = weather_yvr.reset_index(drop=True)
weather_yvr['Region']='YVR'

In [260]:
# fill in the nans
weather_yvr['temp_missing'] = weather_yvr['temperature'].isna().astype(int)
weather_yvr['humid_missing'] = weather_yvr['humidity'].isna().astype(int)
weather_yvr['press_missing'] = weather_yvr['pressure'].isna().astype(int)

column_list = ['temperature', 'humidity', 'pressure']

for column in column_list:
    selected_weather_feature = weather_yvr[column]

    # average of the hour before and after
    weather_feature = (selected_weather_feature.shift(1) + selected_weather_feature.shift(-1)) / 2

    # if weather_feature is still null, average of the same hour one day before and after
    weather_feature = weather_feature.fillna((selected_weather_feature.shift(24) + selected_weather_feature.shift(-24)) / 2)

    # if weather_feature is still null, copy the same hour from the day before
    weather_feature = weather_feature.fillna(selected_weather_feature)

    # if weather_feature is still null, copy the same hour from the day after
    weather_feature = weather_feature.fillna(selected_weather_feature.shift(-24))

    # write back: only the original NaN rows get a new value
    weather_yvr[column] = selected_weather_feature.fillna(weather_feature.round(2))

In [261]:
weather_yvr['weather_missing'] = weather_yvr['weather'].isna().astype(int)
weather_yvr['weather'] = weather_yvr['weather'].ffill() # fill the current null with the previous non-null value

In [262]:
display(weather_yvr)

,date,hour,temperature,humidity,pressure,weather,Region,temp_missing,humid_missing,press_missing,weather_missing
0,2012-01-01,1,2.4,74.0,102.63,Cloudy,YVR,0,0,0,0
1,2012-01-01,2,2.8,69.0,102.62,Cloudy,YVR,0,0,0,0
2,2012-01-01,3,2.8,69.0,102.69,Cloudy,YVR,0,0,0,0
3,2012-01-01,4,2.6,71.0,102.75,Cloudy,YVR,0,0,0,0
4,2012-01-01,5,2.8,69.0,102.78,Cloudy,YVR,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...
73339,2020-05-13,20,15.0,68.0,101.10,Mostly Cloudy,YVR,0,0,0,0
73340,2020-05-13,21,13.4,78.0,101.14,Mostly Cloudy,YVR,0,0,0,1
73341,2020-05-13,22,13.6,73.0,101.19,Mostly Cloudy,YVR,0,0,0,1
73342,2020-05-13,23,12.7,89.0,101.21,Mainly Clear,YVR,0,0,0,0


In [263]:
# Load WYJ weather data
weather_wyj = pd.read_csv('data/Weather_WYJ.csv')

In [264]:
# clean up WYJ data
weather_wyj['datetime'] = pd.to_datetime(weather_wyj['date'])+pd.to_timedelta(weather_wyj['hour']-1, unit='h') # get the date time from data
weather_wyj = weather_wyj.set_index('datetime') # make datetime as the index of the dataframe
weather_wyj = weather_wyj.reindex(pd.date_range(weather_wyj.index.min(), weather_wyj.index.max(), freq='h')) # build a complete hour between the start date and end date. If there is any missing hour or day, it will assign null to the daate and hour fields
weather_wyj['date'] = weather_wyj.index.strftime("%Y-%m-%d") # reassign the date in case there is any missing date
weather_wyj['hour'] = weather_wyj.index.hour+1 # reassign the hour in case there is any missing hour and +1 is added so that the hour start with 1 rather than 0
weather_wyj = weather_wyj.reset_index(drop=True)
weather_wyj['Region']='WYJ'

In [265]:
# fill in the nans
weather_wyj['temp_missing'] = weather_wyj['temperature'].isna().astype(int)
weather_wyj['humid_missing'] = weather_wyj['humidity'].isna().astype(int)
weather_wyj['press_missing'] = weather_wyj['pressure'].isna().astype(int)

column_list = ['temperature', 'humidity', 'pressure']

for column in column_list:
    selected_weather_feature = weather_wyj[column]

    # average of the hour before and after
    weather_feature = (selected_weather_feature.shift(1) + selected_weather_feature.shift(-1)) / 2

    # if weather_feature is still null, average of the same hour one day before and after
    weather_feature = weather_feature.fillna((selected_weather_feature.shift(24) + selected_weather_feature.shift(-24)) / 2)

    # if weather_feature is still null, copy the same hour from the day before
    weather_feature = weather_feature.fillna(selected_weather_feature)

    # if weather_feature is still null, copy the same hour from the day after
    weather_feature = weather_feature.fillna(selected_weather_feature.shift(-24))

    # write back: only the original NaN rows get a new value
    weather_wyj[column] = selected_weather_feature.fillna(weather_feature.round(2))

In [266]:
weather_wyj['weather_missing'] = weather_wyj['weather'].isna().astype(int)
weather_wyj['weather'] = weather_wyj['weather'].ffill() # fill the current null with the previous non-null value

"""
    I notice that weather_wyj has null for all cells in column: temperature, humidity, and pressure. 
    So maybe we can just copy from weather_yvr instead
"""

# use weather data from yvr region
lookup = weather_yvr.set_index(['date', 'hour'])['weather']
weather_wyj['weather'] = pd.Series(list(zip(weather_wyj['date'], weather_wyj['hour']))).map(lookup)

In [267]:
display(weather_wyj)

,date,hour,temperature,humidity,pressure,weather,Region,temp_missing,humid_missing,press_missing,weather_missing
0,2015-01-01,1,0.7,88.0,102.31,Clear,WYJ,0,0,0,1
1,2015-01-01,2,1.0,88.0,102.26,Clear,WYJ,0,0,0,1
2,2015-01-01,3,0.7,90.0,102.22,Clear,WYJ,0,0,0,1
3,2015-01-01,4,0.6,90.0,102.18,Clear,WYJ,0,0,0,1
4,2015-01-01,5,-0.4,91.0,102.11,Clear,WYJ,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...
43819,2019-12-31,20,11.5,98.0,99.37,Cloudy,WYJ,0,0,0,1
43820,2019-12-31,21,11.1,95.0,99.35,Cloudy,WYJ,0,0,0,1
43821,2019-12-31,22,10.8,94.0,99.34,Cloudy,WYJ,0,0,0,1
43822,2019-12-31,23,10.7,92.0,99.38,Mostly Cloudy,WYJ,0,0,0,1


In [268]:
weather_data = pd.concat([weather_wyj, weather_yvr])
display(weather_data)

,date,hour,temperature,humidity,pressure,weather,Region,temp_missing,humid_missing,press_missing,weather_missing
0,2015-01-01,1,0.7,88.0,102.31,Clear,WYJ,0,0,0,1
1,2015-01-01,2,1.0,88.0,102.26,Clear,WYJ,0,0,0,1
2,2015-01-01,3,0.7,90.0,102.22,Clear,WYJ,0,0,0,1
3,2015-01-01,4,0.6,90.0,102.18,Clear,WYJ,0,0,0,1
4,2015-01-01,5,-0.4,91.0,102.11,Clear,WYJ,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...
73339,2020-05-13,20,15.0,68.0,101.10,Mostly Cloudy,YVR,0,0,0,0
73340,2020-05-13,21,13.4,78.0,101.14,Mostly Cloudy,YVR,0,0,0,1
73341,2020-05-13,22,13.6,73.0,101.19,Mostly Cloudy,YVR,0,0,0,1
73342,2020-05-13,23,12.7,89.0,101.21,Mainly Clear,YVR,0,0,0,0
